In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [3]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [5]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [6]:
# ok biolerplate done, now we get to the action:

In [7]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [8]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [9]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

In [10]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = (1/n)*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3264, grad_fn=<NegBackward0>)

In [11]:
# dlogprobs = -logprobs[range(n), Yb].mean() = (-1/n) * [logprobs.sum()]

In [12]:
Yb

tensor([ 8, 14, 15, 22,  0, 19,  9, 14,  5,  1, 20,  3,  8, 14, 12,  0, 11,  0,
        26,  9, 25,  0,  1,  1,  7, 18,  9,  3,  5,  9,  0, 18])

In [13]:
# How to compute the derivative of logprobs correctly?

# loss = -logprobs[range(n), Yb].mean()

# logprobs[range(n), Yb].shape
# loss = -(a, b, c).mean() = -(a+b+c) * 1/3
# dloss/d(a, b, c) = -1/n

# dlogprobs = torch.zeros_like(logprobs) # create a zero matrix with the exact dim as logprobs
# dlogprobs[range(n), Yb] = -1.0/n       # replace the data at the label(Yb) column in each row to (-1.0/n)

In [14]:
# How to calculate dprobs?

# logprobs = probs.log()

# dprobs = dlogprobs_dprobs * dlogprobs
# dprobs = 1.0/probs * dlogprobs

In [15]:
# How to compute the derivative of counts_sum_inv correctly?

# probs = counts * counts_sum_inv

# counts.shape: (32, 27)
# counts_sum_inv.shape: (32, 1)

# How 'counts(32, 27) * counts_sum_inv(32, 1)' works under the hood?
# p = a * b
# a11*b1 a12*b1 a13*b1 ... a127*b1
# a21*b2 a22*b2 a23*b2 ... a227*b2
# .
# .
# .
# a321*b32 a322*b32  ... a3227*b32

# step 1: Transform b: counts_sum_inv (32, 1) => (32, 27) by broadcasting column vector 27 times
# step 2: Multiply counts * counts_sum_inv

# To compute derivative of this operation, we need to compute the derivative for both steps,
# hence, we postpend the '.sum(1, keepdims=True)' expression.
# dcounts_sum_inv = (counts * dprobs).sum(1, keepdims=True)

In [16]:
# How to compute the derivative of dcounts_sum correctly?

# counts_sum_inv = counts_sum**-1

# dloss/dcounts_sum = dcounts_sum_inv/dcounts_sum * dloss/dcounts_sum_inv
# dcounts_sum       = (-1)*counts_sum**-2       * dcounts_sum_inv

In [17]:
# How to compute the derivative of dcounts correctly?

# counts is involved in two expressions, so we need to add the derivatives from both expressions.
# 1. probs = counts * counts_sum_inv
# 2. counts_sum = counts.sum(1, keepdims=True)

# Based on 1st expression, 
# dcounts = counts_sum_inv * dprobs       # dcounts from the 1st branch; use '+=' for other branches of dcounts.

# Based on 2nd expression, 
# counts.shape, counts_sum.shape
# let c = counts; cs = counts_sum;
# cs = c.sum(1, keepdims=True)
# cs1  <-- c11 + c12 + ... + c127
# cs2  <-- c21 + c22 + ... + c227
# .
# .
# cs32 <-- c321 + c322+ .. + c3227

# dloss/dc = dcs/dc * dloss/dcs
# dc = torch.ones_like(c) * dcs           # since it's an addition op along the rows, dcs/dc is 1 across all elements of c.

# dcounts += torch.ones_like(counts) * dcounts_sum 

(torch.Size([32, 27]), torch.Size([32, 1]))

In [23]:
# How to compute the derivative of norm_logits and logit_maxes correctly?

# counts = norm_logits.exp()                                  
# dnorm_logits = counts * dcounts    

# norm_logits = logits - logit_maxes
# dlogit_maxes = dnorm_logits/dlogit_maxes * dloss/dnorm_logits

norm_logits.shape, logits.shape, logit_maxes.shape

# step 1: broadcast logit_maxes from (32, 1) -> (32, 27)
# step 2: Substarct logit_maxes from logits

# derivative of logit_maxes in step 2: (-1.0 * dnorm_logits)
# derivative of logit_maxes in step 1: (-1.0 * dnorm_logits).sum(-1, keepdims=True)
# dlogit_maxes = (-1.0 * dnorm_logits).sum(-1, keepdims=True)
# dlogits = 1.0 * dnorm_logits # 1st Branch of logits

(torch.Size([32, 27]), torch.Size([32, 27]), torch.Size([32, 1]))

In [47]:
# How to compute the derivative of dlogits correctly?

# derivative of 1st branch from the above cell.
# dlogits = 1.0 * dnorm_logits # from 'norm_logits = logits - logit_maxes'

# derivative of 2nd branch of logits:
# logit_maxes = logits.max(1, keepdim=True).values
logit_maxes.shape, logits.shape, logits.max(1,).indices.shape, Yb.shape
# dlogits += dlogit_maxes/dlogits * dloss/dlogit_maxes
# dlogits_2 = torch.zeros_like(logits)
# dlogits_2[range(n), logits.max(1).indices] = 1
# dlogits += dlogits_2 * dlogit_maxes

# OR a one-liner solution for 2nd branch using F.one_hot()
# dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes


(torch.Size([32, 1]), torch.Size([32, 27]), torch.Size([32]), torch.Size([32]))

In [56]:
# How to compute the derivative of dh, dW2 and db2 correctly?
# logits = h @ W2 + b2
dlogits.shape, h.shape, W2.shape, b2.shape
# dh = dlogits @ W2.T # dh.shape  ==  h.shape => (32, 64); dlogits:(32, 27), W2:(64, 27); dlogits @ W2.T
# dW2 = h.T @ dlogits # dW2.shape == W2.shape => (64, 27); dlogits:(32, 27),  h:(32, 64); h.T @ dlogits
# db2 = dlogits.sum(0, keepdims=True)  # db2.shape == b2.shape => (27); dlogits:(32, 27) => (27); dlogits.sum(0, keepdims=True)

(torch.Size([32, 27]),
 torch.Size([32, 64]),
 torch.Size([64, 27]),
 torch.Size([27]))

In [62]:
# How to compute the derivative of bngain, bnraw and bnbias correctly?
# hpreact = bngain * bnraw + bnbias          # .sum(0, keepdims=True) of dbngain and dbnbias is derived from this expression.
hpreact.shape, bngain.shape, bnraw.shape, bnbias.shape
# dbngain = (dhpreact/dbngain) * (dloss/dhpreact)
# dbnraw = (dhpreact/dbnraw) * (dloss/dhpreact)
# dbnbias = (dhpreact/dbnbias) * (dloss/dhpreact)
# dbngain = (bnraw * dhpreact).sum(0, keepdims=True) # .sum(0), bngain:(1, 64) had to broadcast row-wise to (32, 64) for the mul.
# dbnraw = bngain * dhpreact                         # no .sum(0) since bnraw:(32, 64) didn't have to do any transformation.
# dbnbias = (1.0 * dhpreact).sum(0, keepdims=True)   # .sum(0), bnbias:(1, 64) had to broadcast row-wise to (32, 64)

(torch.Size([32, 64]),
 torch.Size([1, 64]),
 torch.Size([32, 64]),
 torch.Size([1, 64]))

In [71]:
# bnraw = bndiff * bnvar_inv
bnraw.shape, bndiff.shape, bnvar_inv.shape
# dbndiff = bnvar_inv * dbnraw
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdims=True)

(torch.Size([32, 64]), torch.Size([32, 64]), torch.Size([1, 64]))

In [77]:
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) 
bnvar.shape, bndiff2.shape, bndiff2.sum(0, keepdim=True).shape
# dbndiff2 = dbnvar/dbndiff2 * dloss/dbnvar
# dbndiff2 = 1.0/(n-1)*torch.ones_like(bndiff2) * dbnvar

(torch.Size([1, 64]), torch.Size([32, 64]), torch.Size([1, 64]))

In [ ]:
# bndiff2 = bndiff**2
# dbndiff += 2.0*bndiff**1 * dbndiff2

In [75]:
# bnvar_inv = (bnvar + 1e-5)**-0.5
bnvar_inv.shape, bnvar.shape
# dbnvar = (dbnvar_inv/dbnvar) * (dloss/dbnvar_inv)
# dbnvar = (-0.5*(bnvar+1e-5)**-1.5) * dbnvar_inv

(torch.Size([1, 64]), torch.Size([1, 64]))

In [78]:
# bndiff = hprebn - bnmeani
hprebn.shape, bnmeani.shape
# dhprebn = 1.0 * dbndiff
# dbnmeani = (-1.0 * dbndiff).sum(0, keepdims=True)

(torch.Size([32, 64]), torch.Size([1, 64]))

In [80]:
# bnmeani = (1/n)*hprebn.sum(0, keepdim=True)
hprebn.shape, hprebn.sum(0, keepdim=True).shape
# dhprebn = (1.0/n)*torch.ones_like(hprebn) * dbnmeani

(torch.Size([32, 64]), torch.Size([1, 64]))

In [82]:
# hprebn = embcat @ W1 + b1
hprebn.shape, embcat.shape, W1.shape, b1.shape
# dembcat = dhprebn @ W1.T
# dW1 = embcat.T @ dhprebn
# db1 = (1.0 * dhprebn).sum(0, keepdims=True)

(torch.Size([32, 64]),
 torch.Size([32, 30]),
 torch.Size([30, 64]),
 torch.Size([64]))

In [85]:
# embcat = emb.view(emb.shape[0], -1)
# emb.shape, emb.view(emb.shape[0], -1).shape
# demb = dembcat.view(emb.shape)

(torch.Size([32, 3, 10]), torch.Size([32, 30]))

In [91]:
# emb = C[Xb]
emb.shape, C.shape
dC = torch.ones_like(C)

(torch.Size([32, 3, 10]), torch.Size([27, 10]))

In [96]:
# Exercise 1: backprop through the whole thing manually, 
# backpropagating through exactly all of the variables 
# as they are defined in the forward pass above, one by one

# loss = -logprobs[range(n), Yb].mean()
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0/n
# logprobs = probs.log()
dprobs = (1.0/probs) * dlogprobs
# probs = counts * counts_sum_inv                             # probs = counts.__mul__(counts_sum_inv)
dcounts_sum_inv = (counts * dprobs).sum(dim=1, keepdims=True) # self.grad  += other.data * out.grad
dcounts = counts_sum_inv * dprobs                             # other.grad += self.data * out.grad
# counts_sum_inv = counts_sum**-1
dcounts_sum = (-1.0)*(counts_sum)**-2.0 * dcounts_sum_inv
# counts_sum = counts.sum(1, keepdims=True)
dcounts += torch.ones_like(counts) * dcounts_sum              # += because it's the 2nd branch of counts.
# counts = norm_logits.exp()                                  # out = self.data.exp()
dnorm_logits = counts * dcounts                               # self.grad = out.data * out.grad
# norm_logits = logits - logit_maxes
dlogit_maxes = (-1.0 * dnorm_logits).sum(-1, keepdims=True)
dlogits = 1.0 * dnorm_logits
# logit_maxes = logits.max(1, keepdim=True).values
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
# logits = h @ W2 + b2
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0, keepdims=True)
# h = torch.tanh(hpreact)
dhpreact = (1.0 - h**2) * dh
# hpreact = bngain * bnraw + bnbias
dbngain = (bnraw * dhpreact).sum(0, keepdims=True)
dbnraw = (bngain * dhpreact)
dbnbias = (1.0 * dhpreact).sum(0, keepdims=True)
# bnraw = bndiff * bnvar_inv
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdims=True)
# bnvar_inv = (bnvar + 1e-5)**-0.5
dbnvar = (-0.5*(bnvar+1e-5)**-1.5) * dbnvar_inv
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) 
dbndiff2 = 1.0/(n-1)*torch.ones_like(bndiff2) * dbnvar
# bndiff2 = bndiff**2
dbndiff += 2.0*bndiff**1 * dbndiff2
# bndiff = hprebn - bnmeani
dhprebn = 1.0 * dbndiff
dbnmeani = (-1.0 * dbndiff).sum(0, keepdims=True)
# bnmeani = (1/n)*hprebn.sum(0, keepdim=True)
dhprebn += (1.0/n)*torch.ones_like(hprebn) * dbnmeani
# hprebn = embcat @ W1 + b1
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = (1.0 * dhprebn).sum(0, keepdims=True)
# embcat = emb.view(emb.shape[0], -1)
demb = dembcat.view(emb.shape)
# emb = C[Xb]
dC = torch.ones_like(C)



cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0
bnraw           | exact: True  | approximate: True  | maxdiff: